# NB_00 — SOURCE_00 Engineering Evidence Extraction

**Source:** *Use of Transition Models to Design High Performance TESs for the LCLS-II Soft X-Ray Spectrometer*  
**Engineering driver:** Absorber Manufacturing  
**Engineering questions:** EQ_02 Manufacturing Tolerances · EQ_04 Detector Variability · EQ_05 Process Validation

This notebook converts the reviewed paper into a structured engineering source record, validates it, and writes the completed canonical YAML plus tabular outputs.


> Run the notebook from top to bottom so repository paths, `scaffold`, extracted evidence, and outputs are created in order.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json, shutil
import pandas as pd
import yaml
pd.set_option("display.max_colwidth", 120)

## 1. Resolve or clone the repository

This notebook supports both:

- a local checkout such as `/home/dan/sensors-becker`;
- Google Colab, where the repository is cloned to `/content/sensors-becker` when it is not already present.

The canonical source record remains:

```text
engineering_navigator/
    absorber_manufacturing/
        source_records/
            SOURCE_00_becker_transition_models.yaml
```

Set `REPO_ROOT_OVERRIDE` only where the repository is stored somewhere else.


In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess


REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE: str | Path | None = None
AUTO_CLONE_IN_COLAB = True


def is_repo_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "engineering_navigator").is_dir()
        and (path / "engineering_navigator" / "absorber_manufacturing").is_dir()
    )


def candidate_repo_roots(start: Path | None = None) -> list[Path]:
    start = (start or Path.cwd()).resolve()
    candidates: list[Path] = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    candidates.extend([start, *start.parents])

    # Common execution locations.
    candidates.extend(
        [
            Path("/content/sensors-becker"),
            Path("/home/dan/sensors-becker"),
            Path.home() / "sensors-becker",
        ]
    )

    # Preserve order while removing duplicates.
    unique: list[Path] = []
    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique.append(candidate)

    return unique


def clone_colab_repository() -> Path | None:
    target = Path("/content/sensors-becker")

    if not AUTO_CLONE_IN_COLAB or not Path("/content").exists():
        return None

    if target.exists() and not is_repo_root(target):
        raise FileExistsError(
            f"{target} exists but is not a sensors-becker repository. "
            "Remove it or set REPO_ROOT_OVERRIDE."
        )

    if not target.exists():
        print(f"Cloning {REPOSITORY_URL} into {target} ...")
        subprocess.run(
            ["git", "clone", REPOSITORY_URL, str(target)],
            check=True,
        )

    return target


def find_repo_root() -> Path:
    for candidate in candidate_repo_roots():
        if is_repo_root(candidate):
            return candidate

    cloned = clone_colab_repository()
    if cloned is not None and is_repo_root(cloned):
        return cloned

    checked = "\n".join(f"  - {path}" for path in candidate_repo_roots())
    raise FileNotFoundError(
        "Could not locate the sensors-becker repository.\n"
        "Checked:\n"
        f"{checked}\n\n"
        "Set REPO_ROOT_OVERRIDE to the repository's absolute path."
    )


REPO_ROOT = find_repo_root()

DRIVER_DIR = (
    REPO_ROOT
    / "engineering_navigator"
    / "absorber_manufacturing"
)

SOURCE_RECORD = (
    DRIVER_DIR
    / "source_records"
    / "SOURCE_00_becker_transition_models.yaml"
)

OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "engineering_questions"
    / "absorber_manufacturing"
    / "SOURCE_00"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not SOURCE_RECORD.exists():
    available = sorted(
        path.name for path in (DRIVER_DIR / "source_records").glob("*.yaml")
    )
    raise FileNotFoundError(
        f"Source record not found: {SOURCE_RECORD}\n"
        f"Available YAML records: {available}"
    )

print(f"Repository root : {REPO_ROOT}")
print(f"Source record   : {SOURCE_RECORD.relative_to(REPO_ROOT)}")
print(f"Output directory: {OUTPUT_DIR.relative_to(REPO_ROOT)}")


## 2. Load the source-record scaffold

This cell loads the canonical YAML record resolved in the previous step.
All later extraction and validation cells depend on `scaffold`.


In [ ]:
with SOURCE_RECORD.open("r", encoding="utf-8") as handle:
    scaffold = yaml.safe_load(handle)

if not isinstance(scaffold, dict):
    raise TypeError(
        f"Expected one top-level YAML mapping in {SOURCE_RECORD}"
    )

required_scaffold_fields = [
    "source_id",
    "title",
    "record_status",
    "extraction_status",
]

missing_scaffold_fields = [
    field
    for field in required_scaffold_fields
    if field not in scaffold
]

if missing_scaffold_fields:
    raise KeyError(
        "Source-record scaffold is missing required fields: "
        + ", ".join(missing_scaffold_fields)
    )

print("Loaded source-record scaffold")
print(f"Source ID         : {scaffold['source_id']}")
print(f"Title             : {scaffold['title']}")
print(f"Record status     : {scaffold['record_status']}")
print(f"Extraction status : {scaffold['extraction_status']}")


## 3. Source-supported engineering extraction

Page references use the manuscript page numbers visible in the PDF.

In [ ]:
authors=["Kelsey M. Morgan","Dan T. Becker","Douglas A. Bennett","William B. Doriese","Johnathon D. Gard","Kent D. Irwin","Sang Jun Lee","Dale Li","John A. B. Mates","Christine G. Pappas","Dan R. Schmidt","Charles J. Titus","Dan D. Van Winkle","Joel N. Ullom","Abigail Wessels","Daniel S. Swetz"]
materials=[
{"name":"Mo/Cu bilayer","role":"TES film","source_pages":[5,8,10]},
{"name":"SiNx membrane","role":"1 µm thermal-isolation membrane","source_pages":[3]},
{"name":"Cu normal-metal bars","role":"transition and critical-current engineering","source_pages":[3,6,10]},
{"name":"Cu banks","role":"prevent superconducting shorts","source_pages":[10]},
{"name":"Mo leads","role":"superconducting leads","source_pages":[10]},
{"name":"Evaporated bismuth","role":"x-ray absorber","source_pages":[6,10]}]
fabrication_methods=[
{"method":"Mo/Cu bilayer TES fabrication on suspended SiNx membrane","purpose":"thermally isolated sensor","source_pages":[3,10]},
{"method":"Patterned Cu bars and banks","purpose":"transition engineering and short prevention","source_pages":[3,6,10]},
{"method":"Evaporated bismuth deposition","purpose":"increase x-ray absorption efficiency","source_pages":[6,10]},
{"method":"Multiple geometries fabricated on one test array","purpose":"test scaling of C and G","source_pages":[6]}]
design_variables=[
{"id":"Tc","name":"critical temperature","unit":"mK"},{"id":"TES_geometry","name":"TES geometry","unit":"µm"},{"id":"bar_count","name":"Cu bar count","unit":"count"},{"id":"bar_spacing","name":"Cu bar spacing","unit":"µm"},{"id":"Bi_thickness","name":"bismuth thickness","unit":"µm"},{"id":"C","name":"heat capacity","unit":"pJ/K"},{"id":"G","name":"thermal conductance","unit":"pW/K"},{"id":"n","name":"thermal exponent","unit":"dimensionless"},{"id":"Rn","name":"normal resistance","unit":"mΩ"},{"id":"alpha","name":"temperature sensitivity","unit":"dimensionless"},{"id":"beta","name":"current sensitivity","unit":"dimensionless"},{"id":"pulse_tau","name":"pulse decay time","unit":"µs"},{"id":"delta_E","name":"energy resolution FWHM","unit":"eV"},{"id":"linear_range","name":"linear energy range","unit":"relative"}]
len(materials),len(design_variables)

In [ ]:
reported_values=[
{"object":"LCLS-II specification","variable":"pixel_count","value":1000,"unit":"pixels","source_page":2},
{"object":"LCLS-II specification","variable":"delta_E","value":0.5,"unit":"eV FWHM","condition":"below 1 keV","source_page":2},
{"object":"LCLS-II specification","variable":"pulse_tau","value":100,"unit":"µs","comparison":"<","source_page":2},
{"object":"previous device","variable":"geometry","value":"212 × 106","unit":"µm","source_page":5},
{"object":"previous device","variable":"bar_count","value":4,"unit":"count","source_page":5},
{"object":"previous device","variable":"Tc","value":76,"unit":"mK","source_page":5},
{"object":"previous device","variable":"delta_E","value":1.02,"unit":"eV FWHM","condition":"1.25 keV","source_page":5},
{"object":"previous device","variable":"C","value":0.11,"unit":"pJ/K","source_page":5},
{"object":"previous device","variable":"Rn","value":14.5,"unit":"mΩ","source_page":5},
{"object":"previous device","variable":"G","value":130,"unit":"pW/K","source_page":5},
{"object":"previous device","variable":"n","value":3.52,"unit":"dimensionless","source_page":5},
{"object":"200 µm 4-bar device","variable":"Tc","value":54.5,"unit":"mK","source_page":6},
{"object":"200 µm 4-bar device","variable":"G","value":61,"unit":"pW/K","source_page":6},
{"object":"200 µm 4-bar device","variable":"C","value":0.166,"unit":"pJ/K","source_page":6},
{"object":"200 µm 4-bar device","variable":"pulse_tau","value":280,"unit":"µs","condition":"10.5% Rn","source_page":6},
{"object":"200 µm 4-bar device","variable":"critically_damped_tau","value":87,"unit":"µs","source_page":6},
{"object":"200 µm 4-bar device","variable":"delta_E","value":0.87,"uncertainty":0.07,"unit":"eV FWHM","condition":"1.254 keV","source_page":7},
{"object":"180 µm 3-bar device","variable":"delta_E","value":0.75,"uncertainty":0.07,"unit":"eV FWHM","condition":"1.254 keV","source_page":7},
{"object":"180 µm 3-bar device","variable":"delta_E","value":0.94,"uncertainty":0.03,"unit":"eV FWHM","condition":"1.487 keV","source_page":7},
{"object":"proposed next generation","variable":"geometry","value":"220 × 220","unit":"µm","source_page":8},
{"object":"proposed next generation","variable":"Tc","value":30,"unit":"mK","source_page":8},
{"object":"proposed next generation","variable":"predicted_delta_E","value":0.5,"unit":"eV FWHM","condition":"1 keV","source_page":8}]
pd.DataFrame(reported_values)

## 4. Mathematical specifications

In [ ]:
equations=[
{"id":"thermal_conductance","expression":"G(T)=n*kappa*T**(n-1)","display":r"G(T)=n\kappa T^{n-1}","source_page":3},
{"id":"pulse_decay_scaling","expression":"tau proportional_to C/G","display":r"\tau\propto C/G","source_page":3},
{"id":"linear_range_scaling","expression":"linear_range proportional_to C*T/alpha","display":r"E_{linear}\propto CT/\alpha","source_page":3},
{"id":"joule_power_balance","expression":"I**2*R=kappa*(T**n-Tb**n)","display":r"I^2R=\kappa(T^n-T_b^n)","source_page":5},
{"id":"energy_resolution_scaling","expression":"delta_E**2 proportional_to (C*T**2/alpha)*J*(1+2*beta)","display":r"\Delta E^2\propto(CT^2/\alpha)J(1+2\beta)","source_page":5},
{"id":"critical_current_temperature","expression":"Ic=Ic0*(1-T/Tc)**(3/2)","display":r"I_c(T)=I_{c0}(1-T/T_c)^{3/2}","source_page":4}]
pd.DataFrame(equations)[["id","display","source_page"]]

In [ ]:
measured_outcomes=[
{"outcome":"Sub-eV resolution demonstrated","result":"0.75 ± 0.07 eV FWHM at 1.254 keV","device":"180 × 180 µm, 3 bars","source_pages":[7,11]},
{"outcome":"Pulse-time requirement met by critically damped estimate","result":"87 µs","device":"200 × 200 µm, 4 bars","source_pages":[6]},
{"outcome":"Heat capacity exceeded simple scaling","result":"0.166 pJ/K; 18% higher than expected","device":"200 × 200 µm, 4 bars","source_pages":[6,8]},
{"outcome":"Thermal conductance below scaling prediction","result":"61 pW/K; 14% lower than expected","device":"200 × 200 µm, 4 bars","source_pages":[6]},
{"outcome":"Model-guided next design specified","result":"220 × 220 µm, Tc=30 mK, predicted 0.5 eV","device":"proposed","source_pages":[7,8]}]
engineering_relationships=[
{"relationship":"Lower Tc reduces C and G, with G expected to fall faster.","effect":"Pulse decay tends to lengthen because tau scales with C/G.","source_pages":[3]},
{"relationship":"Lower Tc requires co-designed geometry and transition parameters.","effect":"Copying the same geometry would reduce linear range and slow pulses.","source_pages":[3,5]},
{"relationship":"Increasing TES area increases C and collection area.","effect":"Geometry can compensate for lower heat capacity per volume.","source_pages":[5,7,8]},
{"relationship":"Critical current varies proportionally with normal-metal bar spacing.","effect":"Preserving spacing helps preserve critical current.","source_pages":[6]},
{"relationship":"Excess heat capacity may reflect copper behavior or process variability.","effect":"More devices are required before fixing pixel-size scaling.","source_pages":[8]}]
engineering_constraints=[
{"constraint":"energy_resolution","specification":"0.5 eV FWHM below 1 keV","source_pages":[2,7]},
{"constraint":"array_scale","specification":"1000 pixels","source_pages":[2]},
{"constraint":"pulse_decay","specification":"<100 µs","source_pages":[2]},
{"constraint":"linear_range","specification":"adequate through 1 keV","source_pages":[6,7]},
{"constraint":"development_cost","specification":"reduce wafers and tested variants","source_pages":[3]}]
assumptions=[
{"assumption":"Metal heat capacity scales approximately linearly with temperature.","source_pages":[3,5,8]},
{"assumption":"n is typically 3–4 for a thin SiNx membrane.","source_pages":[3]},
{"assumption":"The two-fluid model applies where weak-link effects are minimal.","source_pages":[4]},
{"assumption":"30 mK prediction depends on excess heat capacity not invalidating scaling.","source_pages":[8]}]
future_questions=[
"What fabrication variables caused the 18% excess heat capacity?",
"Does excess heat capacity persist below 55 mK?",
"How repeatable are C, G, alpha, beta, and Tc across devices and wafers?",
"What absorber-thickness tolerance preserves absorption without adding heat capacity?",
"Which geometry and bar-layout tolerances dominate energy-resolution variability?",
"Does the proposed 220 × 220 µm, 30 mK design meet both requirements after fabrication?"]
unreported_variables=["wafer-to-wafer yield","device-to-device distributions","formal geometry tolerances","absorber-thickness variation","process-step attribution for excess heat capacity","rework and failure rates"]

## 5. Assemble and validate the completed source record

In [ ]:
completed=dict(scaffold)
completed.update({"record_status":"evidence_extracted","extraction_status":"complete_for_source_record_v1","authors":authors,"materials":materials,"fabrication_methods":fabrication_methods,"design_variables":design_variables,"reported_values":reported_values,"measured_outcomes":measured_outcomes,"equations":equations,"assumptions":assumptions,"engineering_relationships":engineering_relationships,"engineering_constraints":engineering_constraints,"future_questions":future_questions,"unreported_variables":unreported_variables,"extraction_notes":["Page references use manuscript page numbers.","Formal manufacturing distributions and tolerances are not reported.","Next-generation performance is a model-guided prediction, not a measurement."]})
completed.pop("source_supported_relationships",None)
required=["authors","materials","design_variables","reported_values","measured_outcomes","equations","engineering_relationships","engineering_constraints","future_questions"]
assert all(completed.get(k) for k in required)
assert all("source_page" in x for x in completed["reported_values"])
print("Validation passed")
pd.DataFrame({"section":required,"records":[len(completed[k]) for k in required]})

## 6. Write canonical and tabular outputs

In [ ]:
backup=SOURCE_RECORD.with_name(SOURCE_RECORD.stem+".scaffold.yaml")
if not backup.exists(): shutil.copy2(SOURCE_RECORD,backup)
with SOURCE_RECORD.open("w",encoding="utf-8") as f: yaml.safe_dump(completed,f,sort_keys=False,allow_unicode=True,width=110)
pd.DataFrame(reported_values).to_csv(OUTPUT_DIR/"SOURCE_00_reported_values.csv",index=False)
(OUTPUT_DIR/"SOURCE_00_engineering_relationships.json").write_text(json.dumps(engineering_relationships,indent=2),encoding="utf-8")
print(f"Updated {SOURCE_RECORD.relative_to(REPO_ROOT)}")
print(f"Backup  {backup.relative_to(REPO_ROOT)}")

## 7. Engineering handoff

The completed record now feeds `NB_00_EQ_02_MANUFACTURING_TOLERANCES.ipynb`, where it can be compared with the remaining absorber-manufacturing sources.

*Admissible generalizations trail leading specifications.*